# Chapter 18 — Static vs Dynamic Resource Allocation in Spark

In this chapter, we explore how Apache Spark allocates and manages computing resources (executors, CPU cores, and memory) across a cluster. We compare **Static Resource Allocation** with **Dynamic Resource Allocation (DRA)**, uncover the mechanics of executor scaling, solve the shuffle-data dilemma with **Shuffle Tracking** and the **External Shuffle Service (ESS)**, and distinguish **Spark Dynamic Allocation** from **Databricks / Cloud Cluster Autoscaling**.

---

## 📝 1. The Resource Allocation Problem in Spark

**When you submit a Spark application to a cluster, Spark creates executor JVM processes across worker machines to compute your tasks. But how many executors should your job get, and how long should they stay alive?**

```text
┌──────────────────────────────────────────────────────────────────┐
│                   HOW SPARK RUNS YOUR APPLICATION                │
├──────────────────────────────────────────────────────────────────┤
│ (1) STATIC ALLOCATION          │ (2) DYNAMIC ALLOCATION (DRA)    │
│                                │                                 │
│  [Driver]                      │  [Driver]                       │
│     │                          │     │                           │
│     ▼ (Allocates fixed 10)     │     ▼ (Asks as work grows)      │
│  ┌──────────────────────────┐  │  ┌───────────────────────────┐  │
│  │ 10 Executors Locked In   │  │  │ Min: 1  --->  Max: 10     │  │
│  │ Active or Idle: Always 10│  │  │ Scales up when busy       │  │
│  │ Cluster cannot reclaim   │  │  │ Scales down when idle     │  │
│  └──────────────────────────┘  │  └───────────────────────────┘  │
└──────────────────────────────────────────────────────────────────┘
```

What the picture is saying, side by side:

1. **(1) Static Allocation:** You request a fixed number of executors up front (e.g., 10 executors). Spark grabs all 10 at application launch and holds onto them until the entire application shuts down — whether those executors are actively crunching data or sitting 100% idle.
2. **(2) Dynamic Resource Allocation (DRA):** You give Spark a range (e.g., Min: 1, Max: 10). Spark starts with a small baseline, requests more executors when a queue of tasks builds up, and gives idle executors back to the cluster manager when the heavy work is finished.

---

## 📝 2. Static Resource Allocation: How It Works & The Wastage Problem

**In Static Allocation, resource boundaries are rigid. You set `spark.executor.instances` (or pass `--num-executors`), and that number stays locked for the lifetime of your `SparkSession`.**

Consider a real-world 3-stage data pipeline running with a static allocation of 10 executors:

```text
┌──────────────────────────────────────────────────────────────────┐
│          STATIC ALLOCATION: 3 STAGES OF A SPARK JOB              │
├──────────────────────────────────────────────────────────────────┤
│                                                                  │
│  Stage 1: Heavy Scan & Filter (Needs 10 Executors)               │
│  ┌───┬───┬───┬───┬───┬───┬───┬───┬───┬───┐                       │
│  │ 1 │ 2 │ 3 │ 4 │ 5 │ 6 │ 7 │ 8 │ 9 │10 │ ---> 100% Busy       │
│  └───┴───┴───┴───┴───┴───┴───┴───┴───┴───┘                       │
│                                                                  │
│  Stage 2: Small Lookup / Aggregation (Needs 1 Executor)          │
│  ┌───┐ ┌───┬───┬───┬───┬───┬───┬───┬───┬───┐                     │
│  │ 1 │ │ 2 │ 3 │ 4 │ 5 │ 6 │ 7 │ 8 │ 9 │10 │                     │
│  └───┘ └───┴───┴───┴───┴───┴───┴───┴───┴───┘                     │
│  Busy  └────────────── 9 Idle Executors (90% Wasted!) ─────────┘ │
│                                                                  │
│  Stage 3: Partition Write (Needs 4 Executors)                    │
│  ┌───┬───┬───┬───┐ ┌───┬───┬───┬───┬───┬───┐                     │
│  │ 1 │ 2 │ 3 │ 4 │ │ 5 │ 6 │ 7 │ 8 │ 9 │10 │                     │
│  └───┴───┴───┴───┘ └───┴───┴───┴───┴───┴───┘                     │
│  Busy (4)          └────── 6 Idle Executors (60% Wasted) ──────┘ │
│                                                                  │
└──────────────────────────────────────────────────────────────────┘
```

Why this model creates severe operational problems:

1. **Idle Resource Wastage:** In Stage 2, only 1 executor is needed. The other 9 executors sit completely idle, yet they consume cluster memory, CPU cores, and cloud money.
2. **Resource Starvation (Multi-Tenancy):** In a shared corporate cluster (e.g. YARN or Kubernetes), if Application A locks 50 idle executors, Application B submitted by another team gets stuck in a `PENDING` queue waiting for free resources.
3. **Inability to Handle Data Surges:** If tomorrow's incoming dataset is 5x larger, your static configuration cannot expand, causing tasks to queue up and processing time to explode.

---

In [7]:
# SparkSession configured with Static Allocation (Fixed 4 Executors)
from pyspark.sql import SparkSession

spark_static = (
    SparkSession.builder
    .appName("Static Resource Allocation Demo")
    .master("local[*]")
    .config("spark.executor.instances", 4)
    .config("spark.executor.cores", 2)
    .config("spark.executor.memory", "1g")
    .config("spark.dynamicAllocation.enabled", "false")
    .getOrCreate()
)

spark_static

#### ☝️ Static Configuration — Fixed From Start to Finish

That cell configured a Spark application with static allocation:
- `spark.dynamicAllocation.enabled = false`: Disables automatic scaling.
- `spark.executor.instances = 4`: Locks in exactly 4 executors for the entire duration.
- Even if your code does nothing for 3 hours, those 4 executors remain allocated.

---

## 📝 3. Dynamic Resource Allocation (DRA): The Elastic Approach

**Dynamic Resource Allocation allows a Spark application to request executors when tasks are waiting in the backlog, and return executors when they sit idle.**

Here is the exact lifecycle of how Spark scales up and down:

```text
┌──────────────────────────────────────────────────────────────────┐
│             HOW DYNAMIC RESOURCE ALLOCATION (DRA) WORKS          │
├──────────────────────────────────────────────────────────────────┤
│                                                                  │
│  (1) Job Starts with Initial Executors                           │
│      ┌─────────────────────────┐                                 │
│      │ 2 Initial Executors     │                                 │
│      └─────────────────────────┘                                 │
│                   │                                              │
│                   ▼                                              │
│  (2) Task Backlog Detected (schedulerBacklogTimeout: 1s)         │
│      ┌─────────────────────────┐                                 │
│      │ Tasks queued > 1 sec    │                                 │
│      └─────────────────────────┘                                 │
│                   │                                              │
│                   ▼                                              │
│  (3) Exponential Scale-Up (1 ---> 2 ---> 4 ---> 8 Executors)     │
│      ┌─────────────────────────────────────────┐                 │
│      │ Spark requests executors from Manager   │                 │
│      │ Caps at maxExecutors (e.g. 10)          │                 │
│      └─────────────────────────────────────────┘                 │
│                   │                                              │
│                   ▼                                              │
│  (4) Idle Timeout & Scale-Down (executorIdleTimeout: 60s)        │
│      ┌─────────────────────────────────────────┐                 │
│      │ Executor has no tasks for 60s           │                 │
│      │ Released back to cluster (down to min)  │                 │
│      └─────────────────────────────────────────┘                 │
│                                                                  │
└──────────────────────────────────────────────────────────────────┘
```

The four steps, in words:

1. **(1) Initial Allocation:** Spark launches with `spark.dynamicAllocation.initialExecutors` (e.g., 2).
2. **(2) Backlog Detection:** When a job submits tasks and there are more tasks than available executor slots, tasks wait in the queue. If tasks remain queued for `spark.dynamicAllocation.schedulerBacklogTimeout` (default: 1 second), Spark triggers a scale-up request.
3. **(3) Exponential Scale-Up:** Spark does not request all executors at once. It uses an **exponential policy**: first 1 executor, then if backlog persists for `sustainedSchedulerBacklogTimeout`, it requests 2, then 4, 8, 16... doubling each round until the backlog is cleared or `spark.dynamicAllocation.maxExecutors` is reached.
4. **(4) Scale-Down on Idle:** When a stage finishes and an executor has no active tasks for `spark.dynamicAllocation.executorIdleTimeout` (default: 60 seconds), Spark decommissions that executor and returns its memory and CPU back to the cluster manager. It will never drop below `spark.dynamicAllocation.minExecutors`.

### Key Configuration Parameters for DRA:

| Configuration Parameter | Default | Purpose |
|---|---|---|
| `spark.dynamicAllocation.enabled` | `false` | Master switch to enable Dynamic Resource Allocation |
| `spark.dynamicAllocation.minExecutors` | `0` | Lower bound of executors the app will ever hold |
| `spark.dynamicAllocation.maxExecutors` | `infinity` | Upper bound ceiling of executors to prevent runaway usage |
| `spark.dynamicAllocation.initialExecutors` | `minExecutors` | Number of executors allocated at startup |
| `spark.dynamicAllocation.schedulerBacklogTimeout` | `1s` | Time pending tasks must queue before scale-up starts |
| `spark.dynamicAllocation.sustainedSchedulerBacklogTimeout` | `1s` | Time between consecutive scale-up rounds |
| `spark.dynamicAllocation.executorIdleTimeout` | `60s` | Idle time before an unused executor is decommissioned |
| `spark.dynamicAllocation.cachedExecutorIdleTimeout` | `infinity` | Idle time before an executor holding cached RDD/DataFrame is killed |

---

## 📝 4. The Shuffle Dilemma: Why Scaling Down Was Hard (And How Spark Solved It)

**In earlier Spark versions, dynamically killing an executor was dangerous because executors store shuffle files on their local disks.**

```text
┌──────────────────────────────────────────────────────────────────┐
│              THE SHUFFLE DILEMMA & THE TWO SOLUTIONS             │
├──────────────────────────────────────────────────────────────────┤
│                                                                  │
│  (1) THE PROBLEM: Executor Killed ---> Local Shuffle Files Lost  │
│  ┌─────────────────────────┐                                     │
│  │ Executor A (IDLE)       │ ---> Spark kills Executor A         │
│  │ Local Disk: [Shuffle]   │ ---> [Shuffle Files Lost!]          │
│  └─────────────────────────┘      Stage 2: FetchFailedException! │
│                                                                  │
│  (2) SOLUTION 1: External Shuffle Service (ESS)                  │
│  ┌────────────────────────────────────────────────────────────┐  │
│  │ Worker Machine                                             │  │
│  │  ┌────────────────────────┐    ┌────────────────────────┐  │  │
│  │  │ Executor A (Killed)    │    │ ESS Daemon (Keeps Run) │  │  │
│  │  └────────────────────────┘    │ Reads & serves shuffle │  │  │
│  │  Local Disk: [Shuffle Files] <───┘                         │  │
│  └────────────────────────────────────────────────────────────┘  │
│                                                                  │
│  (3) SOLUTION 2: Shuffle Tracking (Spark 3.0+ Built-in)          │
│  ┌────────────────────────────────────────────────────────────┐  │
│  │ Driver tracks shuffle files per executor.                  │  │
│  │ Decommission is deferred until shuffle data is GC'd!       │  │
│  └────────────────────────────────────────────────────────────┘  │
│                                                                  │
└──────────────────────────────────────────────────────────────────┘
```

The three parts of the shuffle story:

1. **(1) The Problem (Shuffle File Loss):** In Chapter 14, we learned that a `groupBy` or `join` writes intermediate shuffle files to the executor's local disk. Downstream tasks in Stage 2 must fetch those files over the network. If Spark kills Executor A because it became idle, its local disk shuffle files are deleted! When Stage 2 tasks try to fetch those files, they throw a `FetchFailedException`, forcing Spark to rerun the entire previous stage.
2. **(2) Solution 1: External Shuffle Service (ESS):** Set `spark.shuffle.service.enabled = true`. An auxiliary background daemon runs on each worker node independent of any Spark executor. When Executor A writes shuffle files, ESS serves them to downstream executors even after Executor A has been terminated.
3. **(3) Solution 2: Shuffle Tracking (Spark 3.0+):** Set `spark.dynamicAllocation.shuffleTracking.enabled = true`. Spark's driver monitors which executors hold active shuffle outputs. Spark will safely keep those executors alive (or wait until the shuffle files are no longer needed) before releasing them. This eliminates the need to run an external shuffle service daemon — essential for Kubernetes and containerized clusters!

---

## 📝 5. Spark Dynamic Allocation vs Databricks / Cloud Autoscaling

**A common interview and architecture confusion: both are called "autoscaling", but they operate at completely different layers of the infrastructure.**

```text
┌──────────────────────────────────────────────────────────────────┐
│         TWO DISTINCT LAYERS OF AUTOSCALING IN THE CLOUD          │
├──────────────────────────────────────────────────────────────────┤
│                                                                  │
│  (1) LAYER 1: INFRASTRUCTURE AUTOSCALER (Databricks / Cloud VMs) │
│  ┌────────────────────────────────────────────────────────────┐  │
│  │ Provisions / Terminates Virtual Machines (EC2 / Azure VMs) │  │
│  │ [Worker VM 1]   [Worker VM 2]   [Worker VM 3 (Spun Up)]    │  │
│  └────────────────────────────────────────────────────────────┘  │
│                               │                                  │
│                               ▼                                  │
│  (2) LAYER 2: SPARK DYNAMIC RESOURCE ALLOCATION (Inside Spark)   │
│  ┌────────────────────────────────────────────────────────────┐  │
│  │ Creates / Terminates Spark Executor JVMs inside active VMs │  │
│  │ [Exec 1 (JVM)]   [Exec 2 (JVM)]   [Exec 3 (JVM)]           │  │
│  └────────────────────────────────────────────────────────────┘  │
│                                                                  │
│  (3) Databricks Optimized Autoscaling: Coordinates Layer 1 & 2!  │
│                                                                  │
└──────────────────────────────────────────────────────────────────┘
```

The two layers, explained:

1. **(1) Layer 1: Infrastructure / Node Level Autoscaling (Cloud / Databricks):**
   - **What it scales:** Cloud Virtual Machines (e.g. AWS EC2, Azure VMs, GCP Compute Engine).
   - **How it works:** When CPU or memory utilization on worker VMs exceeds 80%, Databricks requests additional VM instances from AWS/Azure. When VMs sit unused for a threshold (e.g., 15 mins), it terminates the VM to stop cloud billing.
   - **Scale Time:** Takes **2 to 5 minutes** to provision and boot up a new VM.

2. **(2) Layer 2: Application / Process Level Autoscaling (Spark DRA):**
   - **What it scales:** Spark Executor JVM processes inside already-running VMs.
   - **How it works:** When tasks queue up in the Spark scheduler, Spark launches new executor processes on existing cluster nodes.
   - **Scale Time:** Takes **milliseconds to seconds** because the hardware is already booted.

3. **(3) Databricks Optimized Autoscaling (Best of Both):**
   - Databricks deeply integrates both layers. If Spark tasks queue up and the existing VMs are already full of executors, Databricks triggers Layer 1 (adds new worker VMs) and immediately launches Layer 2 (Spark executors on those new VMs).

---

In [8]:
# Initialize SparkSession with Dynamic Resource Allocation & Shuffle Tracking
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Dynamic Resource Allocation in Action")
    .master("local[*]")
    .config("spark.dynamicAllocation.enabled", "true")
    .config("spark.dynamicAllocation.shuffleTracking.enabled", "true")
    .config("spark.dynamicAllocation.minExecutors", "1")
    .config("spark.dynamicAllocation.maxExecutors", "8")
    .config("spark.dynamicAllocation.initialExecutors", "2")
    .config("spark.dynamicAllocation.executorIdleTimeout", "60s")
    .config("spark.dynamicAllocation.schedulerBacklogTimeout", "1s")
    .getOrCreate()
)

spark

#### ☝️ Creating a Session with DRA and Shuffle Tracking

In this session configuration:
- `spark.dynamicAllocation.enabled = true`: Enables dynamic scaling.
- `spark.dynamicAllocation.shuffleTracking.enabled = true`: Activates Spark 3.0+ shuffle tracking so executors with shuffle data are not unsafely killed.
- `minExecutors = 1` and `maxExecutors = 8`: Defines our elastic boundaries.

---

In [9]:
# Verify Active Dynamic Allocation Properties
dra_keys = [
    "spark.dynamicAllocation.enabled",
    "spark.dynamicAllocation.shuffleTracking.enabled",
    "spark.dynamicAllocation.minExecutors",
    "spark.dynamicAllocation.maxExecutors",
    "spark.dynamicAllocation.initialExecutors",
    "spark.dynamicAllocation.executorIdleTimeout",
    "spark.dynamicAllocation.schedulerBacklogTimeout",
]

print("=== Active Dynamic Allocation Configurations ===")
for key in dra_keys:
    value = spark.conf.get(key, "Not Set")
    print(f"{key:52} = {value}")

=== Active Dynamic Allocation Configurations ===
spark.dynamicAllocation.enabled                      = true
spark.dynamicAllocation.shuffleTracking.enabled      = true
spark.dynamicAllocation.minExecutors                 = 1
spark.dynamicAllocation.maxExecutors                 = 8
spark.dynamicAllocation.initialExecutors             = 2
spark.dynamicAllocation.executorIdleTimeout          = 60s
spark.dynamicAllocation.schedulerBacklogTimeout      = 1s


#### ☝️ Checking Active Configurations on the Spark Driver

`spark.conf.get()` reads the active runtime parameters directly from the Spark configuration registry, confirming that our elastic bounds and backlog thresholds are in effect.

---

In [10]:
# Load Employee Dataset (Small Workload Baseline)
emp_schema = "employee_id string, department_id int, name string, age int, gender string, salary double, hire_date string"
emp = spark.read.format("csv").schema(emp_schema).option("header", True).load("data/csv/emp.csv")

print("Employee Row Count:", emp.count())
emp.show(5)

Employee Row Count: 20
+-----------+-------------+----------+---+------+-------+----------+
|employee_id|department_id|      name|age|gender| salary| hire_date|
+-----------+-------------+----------+---+------+-------+----------+
|        001|          101|  John Doe| 30|  Male|50000.0|2015-01-01|
|        002|          101|Jane Smith| 25|Female|45000.0|2016-02-15|
|        003|          102| Bob Brown| 35|  Male|55000.0|2014-05-01|
|        004|          102| Alice Lee| 28|Female|48000.0|2017-09-30|
|        005|          103| Jack Chan| 40|  Male|60000.0|2013-04-01|
+-----------+-------------+----------+---+------+-------+----------+
only showing top 5 rows



#### ☝️ Small Workload — Baseline Resource Consumption

The clean `emp.csv` file has 20 rows and 1 partition. Running `.count()` or `.show()` triggers 1 single task. Because the task backlog is zero, DRA stays at `minExecutors` and does not waste cluster resources requesting unneeded executors.

---

In [11]:
# Wide Transformation with High Partition Count (Simulating Heavy Backlog)
'''SELECT department_id, COUNT(*) as emp_count, AVG(salary) as avg_salary FROM emp GROUP BY department_id'''
from pyspark.sql.functions import avg, count

emp_dept_summary = (
    emp
    .repartition(64, "department_id")
    .groupBy("department_id")
    .agg(
        count("department_id").alias("emp_count"),
        avg("salary").alias("avg_salary")
    )
    .orderBy("department_id")
)

emp_dept_summary.show()

+-------------+---------+----------+
|department_id|emp_count|avg_salary|
+-------------+---------+----------+
|          101|        3|   55000.0|
|          102|        4|   51750.0|
|          103|        4|   58000.0|
|          104|        3|   54000.0|
|          105|        2|   55500.0|
|          106|        2|   69000.0|
|          107|        2|   47500.0|
+-------------+---------+----------+



#### ☝️ Wide Transformation with 64 Partitions — Triggering Task Demand

Here, `.repartition(64, "department_id")` and `groupBy()` force 64 shuffle partitions. When submitted to a real cluster:
1. 64 tasks are created simultaneously.
2. The initial executors cannot process 64 tasks in a single wave, causing tasks to sit in the scheduler queue.
3. Once the queue delay exceeds `schedulerBacklogTimeout` (1s), DRA dynamically requests additional executors up to `maxExecutors`.
4. After aggregation completes and tasks finish, executors idle for 60s and scale back down.

---

In [12]:
# Inspecting Runtime Context & Execution Status
sc = spark.sparkContext
status_tracker = sc.statusTracker()

print("Application ID       :", sc.applicationId)
print("Default Parallelism  :", sc.defaultParallelism)
print("Active Job IDs       :", status_tracker.getActiveJobsIds())
print("Active Stage IDs     :", status_tracker.getActiveStageIds())

Application ID       : local-1787993534049
Default Parallelism  : 20
Active Job IDs       : []
Active Stage IDs     : []


#### ☝️ Monitoring Execution State via Status Tracker

`sc.statusTracker()` gives programmatic access to active jobs, stages, and executor metrics, reflecting what the Spark UI (`http://localhost:4040/executors/`) displays in real time.

---

## 📝 6. Static vs Dynamic Allocation: Decision Matrix & Best Practices

**When should you choose Static Allocation, and when should you choose Dynamic Allocation?**

### Comparison Matrix

| Feature | Static Allocation | Dynamic Resource Allocation (DRA) |
|---|---|---|
| **Resource Control** | Pre-allocated, fixed at launch | Elastic, scales based on task queue demand |
| **Cluster Utilization** | Low to Medium (idle executors hold onto memory/cores) | High (executors released when unused) |
| **Best Workload Type** | Predictable, single-tenant, strict SLA batch/streaming | Variable, multi-stage pipelines, interactive notebooks |
| **Multi-Tenancy Impact** | Risk of resource starvation for other jobs | Enables fair sharing across teams & applications |
| **Shuffle Dependency** | No special shuffle requirement | Needs **Shuffle Tracking** or **External Shuffle Service** |
| **Cost Efficiency** | Lower on shared cloud clusters (wastes idle capacity) | Maximum cost efficiency on cloud & on-prem |

---

### Golden Rules & Best Practices:

1. **Always set `maxExecutors` when using DRA:** Leaving `maxExecutors` unset defaults to infinity, which allows a single runaway job with 10,000 partitions to consume the entire cluster and starve all other users.
2. **Enable Shuffle Tracking in Spark 3.0+:** Always set `spark.dynamicAllocation.shuffleTracking.enabled = true` on Kubernetes or Standalone clusters so you do not need to install complex External Shuffle Service daemons.
3. **Use Static Allocation for Low-Latency Streaming:** For 24/7 Spark Structured Streaming jobs with sub-second micro-batch intervals, constant executor scale-up and scale-down causes overhead and latency spikes. Fixed static executors provide predictable SLA.
4. **Use Dynamic Allocation for Interactive Notebooks:** Jupyter and Databricks notebooks spend 90% of their time waiting for the user to think and type. DRA releases executors during thinking time, preventing idle clusters from burning money.

---

In [ ]:
spark.stop()
spark_static.stop()